# DyTopo — Real Dataset Reproduction

**Paper:** Lu et al. 2026 — *DyTopo: Dynamic Topology Routing for Multi-Agent Reasoning via Semantic Matching* (arXiv:2602.06039)

## Datasets used in the paper (Table 1)

| Dataset | Domain | Difficulty | Paper samples | HuggingFace ID | License |
|---------|--------|-----------|--------------|----------------|---------|
| HumanEval | Code | Fundamental | 164 (all) | `openai/openai_humaneval` | MIT |
| APPS-Competition | Code | Advanced | 100 of 1,000 | `codeparrot/apps` | MIT |
| MATH-500 | Math | Hard | 500 (all) | `HuggingFaceH4/MATH-500` | MIT |
| Omni-MATH | Math | Expert | 70 (7 domains × 10) | `KbsdJames/Omni-MATH` | Apache 2.0 |

## Local environment feasibility assessment

**All 4 datasets are freely downloadable — no login, no gating.** The practical constraints are:

1. **HumanEval** — 164 problems, ~130 KB. No friction. Metric: execution-based pass@1.
2. **APPS-Competition** — 1,000 competition problems (~1.3 GB total, ~130 MB competition split). HuggingFace viewer disabled but `load_dataset()` works. Metric: execution accuracy against hidden test cases. **Requires a Python sandbox** (subprocess/docker) to execute generated code safely.
3. **MATH-500** — 500 problems, <1 MB. No friction. Metric: exact-match (lightweight string normalizer).
4. **Omni-MATH** — 4,428 total, we sample 70. ~10 MB. Canonical metric needs GPT-4o judge; we use the rule-based fallback (no API cost).

**LLM requirement:** The paper uses `qwen3:8b` (via vLLM). Locally we use **Ollama** with `qwen2.5:7b` or `llama3.1:8b` — both are available free.

This notebook:
- Section 1: Verifies all dataset downloads work
- Section 2: Explores each dataset's structure
- Section 3: Runs DyTopo on a small sample (N=5 problems per dataset) in **mock mode** for rapid iteration
- Section 4: Sets up the real evaluation harness (real LLM via Ollama)
- Section 5: Full feasibility report for your local machine

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install datasets sentence-transformers httpx numpy matplotlib pandas tqdm
# For APPS code execution: subprocess is stdlib (no extra install)

import sys, os, re, json, asyncio, subprocess, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

API_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'api')
if API_DIR not in sys.path:
    sys.path.insert(0, API_DIR)

from dytopo_engine import DyTopoConfig, run_dytopo

print('Python:', sys.version.split()[0])
print('API dir:', API_DIR)

# Output dir for cached samples
DATA_CACHE = Path('dytopo_dataset_cache')
DATA_CACHE.mkdir(exist_ok=True)
print('Cache dir:', DATA_CACHE.resolve())

---
## Section 1 — Dataset Download & Feasibility Check

In [ ]:
import importlib, time

def check_pkg(name, import_name=None):
    try:
        importlib.import_module(import_name or name)
        print(f'  ✓ {name}')
        return True
    except ImportError:
        print(f'  ✗ {name}  →  pip install {name}')
        return False

print('Required packages:')
ok = all([
    check_pkg('datasets'),
    check_pkg('sentence-transformers', 'sentence_transformers'),
    check_pkg('httpx'),
    check_pkg('tqdm'),
])
if not ok:
    raise SystemExit('Install missing packages above, then re-run.')

In [ ]:
from datasets import load_dataset

# ── 1. HumanEval ─────────────────────────────────────────────────────────────
print('Loading HumanEval...')
t0 = time.time()
humaneval_ds = load_dataset('openai/openai_humaneval', split='test', trust_remote_code=True)
print(f'  ✓  {len(humaneval_ds)} problems  ({time.time()-t0:.1f}s)')
print(f'  Columns: {humaneval_ds.column_names}')
print(f'  Sample task_id: {humaneval_ds[0]["task_id"]}')

In [ ]:
# ── 2. APPS-Competition ───────────────────────────────────────────────────────
# Note: viewer is disabled on HuggingFace but load_dataset works fine locally
print('Loading APPS (competition split)...')
print('  (First run downloads ~1.3 GB — subsequent runs use cache)')
t0 = time.time()
try:
    apps_ds_all = load_dataset('codeparrot/apps', split='test', trust_remote_code=True)
    apps_ds = apps_ds_all.filter(lambda x: x['difficulty'] == 'competition')
    print(f'  ✓  {len(apps_ds)} competition problems  ({time.time()-t0:.1f}s)')
    print(f'  Columns: {apps_ds.column_names}')
    APPS_OK = True
except Exception as e:
    print(f'  ✗  Download failed: {e}')
    print('  Fallback: will use HumanEval only for code evaluation')
    APPS_OK = False

In [ ]:
# ── 3. MATH-500 ────────────────────────────────────────────────────────────────
print('Loading MATH-500...')
t0 = time.time()
math500_ds = load_dataset('HuggingFaceH4/MATH-500', split='test')
print(f'  ✓  {len(math500_ds)} problems  ({time.time()-t0:.1f}s)')
print(f'  Columns: {math500_ds.column_names}')
subjects = pd.Series([x['subject'] for x in math500_ds]).value_counts()
print('  Subject distribution:')
for subj, cnt in subjects.items():
    print(f'    {subj}: {cnt}')

In [ ]:
# ── 4. Omni-MATH ──────────────────────────────────────────────────────────────
print('Loading Omni-MATH...')
t0 = time.time()
omnimath_ds = load_dataset('KbsdJames/Omni-MATH', split='test')
print(f'  ✓  {len(omnimath_ds)} total problems  ({time.time()-t0:.1f}s)')
print(f'  Columns: {omnimath_ds.column_names}')

# Inspect domain structure
all_domains = [x.get('domain', x.get('subject', 'unknown')) for x in omnimath_ds]
domain_counts = pd.Series(all_domains).value_counts().head(15)
print('  Top domains:')
for d, c in domain_counts.items():
    print(f'    {d}: {c}')

---
## Section 2 — Dataset Structure & Sample Problems

In [ ]:
# ── HumanEval structure ───────────────────────────────────────────────────────
sample = humaneval_ds[0]
print('=== HumanEval sample ===')
print(f'task_id:  {sample["task_id"]}')
print(f'prompt (first 300 chars):')
print(textwrap.indent(sample['prompt'][:300], '  '))
print(f'canonical_solution (first 150 chars):')
print(textwrap.indent(sample['canonical_solution'][:150], '  '))
print(f'test (first 200 chars):')
print(textwrap.indent(str(sample['test'])[:200], '  '))
print(f'entry_point: {sample["entry_point"]}')

In [ ]:
# ── MATH-500 structure ────────────────────────────────────────────────────────
sample_m = math500_ds[0]
print('=== MATH-500 sample ===')
print(f'subject:  {sample_m["subject"]}')
print(f'level:    {sample_m["level"]}')
print(f'problem (first 300 chars):')
print(textwrap.indent(sample_m['problem'][:300], '  '))
print(f'solution (first 200 chars):')
print(textwrap.indent(sample_m['solution'][:200], '  '))
print(f'answer:   {sample_m["answer"]}')

In [ ]:
# ── Omni-MATH structure ────────────────────────────────────────────────────────
sample_o = omnimath_ds[0]
print('=== Omni-MATH sample ===')
print(json.dumps({k: str(v)[:200] for k, v in sample_o.items()}, indent=2))

---
## Section 3 — Build Paper-Aligned Sample Sets

Replicate the exact sampling strategy from the paper:
- **HumanEval**: all 164
- **APPS-Competition**: 100 random from competition split (seed=42)
- **MATH-500**: all 500
- **Omni-MATH**: 70 = 7 domains × 10 per domain (stratified, seed=42)

For this notebook we also create a **mini-sample** (N=5 per dataset) for quick testing.

In [ ]:
import random
random.seed(42)
np.random.seed(42)

# ── HumanEval → prompt + test + entry_point ───────────────────────────────────
humaneval_problems = [
    {
        'id': x['task_id'],
        'problem': x['prompt'],
        'canonical': x['canonical_solution'],
        'test': x['test'],
        'entry_point': x['entry_point'],
        'dataset': 'humaneval',
    }
    for x in humaneval_ds
]
print(f'HumanEval: {len(humaneval_problems)} problems')

# ── APPS-Competition → 100 random ─────────────────────────────────────────────
if APPS_OK:
    apps_idxs = random.sample(range(len(apps_ds)), min(100, len(apps_ds)))
    apps_problems = [
        {
            'id': f'apps_{i}',
            'problem': apps_ds[i]['question'],
            'test_cases': apps_ds[i]['input_output'],
            'solutions': apps_ds[i]['solutions'],
            'dataset': 'apps',
        }
        for i in apps_idxs
    ]
    print(f'APPS-Competition: {len(apps_problems)} problems sampled')
else:
    apps_problems = []
    print('APPS-Competition: skipped (download failed)')

# ── MATH-500 → all 500 ────────────────────────────────────────────────────────
math500_problems = [
    {
        'id': f'math500_{i}',
        'problem': x['problem'],
        'answer': x['answer'],
        'subject': x['subject'],
        'level': x['level'],
        'dataset': 'math500',
    }
    for i, x in enumerate(math500_ds)
]
print(f'MATH-500: {len(math500_problems)} problems')

# ── Omni-MATH → 70 = 7 domains × 10 ──────────────────────────────────────────
def sample_omnimath(ds, n_domains=7, n_per_domain=10, seed=42):
    rng = random.Random(seed)
    # Try both 'domain' and 'subject' columns
    domain_key = 'domain' if 'domain' in ds.column_names else 'subject'
    by_domain = {}
    for i, x in enumerate(ds):
        d = x.get(domain_key, 'unknown')
        # Normalize: take first-level domain (e.g. 'Algebra / Number Theory' → 'Algebra')
        d_top = d.split('/')[0].strip() if d else 'unknown'
        by_domain.setdefault(d_top, []).append((i, x))
    # Select top n_domains by count
    top_domains = sorted(by_domain, key=lambda k: -len(by_domain[k]))[:n_domains]
    print(f'  Selected domains: {top_domains}')
    selected = []
    for dom in top_domains:
        pool = by_domain[dom]
        sample = rng.sample(pool, min(n_per_domain, len(pool)))
        for idx, x in sample:
            selected.append({
                'id': f'omnimath_{idx}',
                'problem': x.get('problem', x.get('question', '')),
                'answer': x.get('answer', x.get('solution', '')),
                'domain': dom,
                'dataset': 'omnimath',
            })
    return selected

omnimath_problems = sample_omnimath(omnimath_ds)
print(f'Omni-MATH: {len(omnimath_problems)} problems sampled')

# ── Mini-samples for quick testing (5 per dataset) ────────────────────────────
mini = {
    'humaneval': random.sample(humaneval_problems, 5),
    'apps':      random.sample(apps_problems, 5) if apps_problems else [],
    'math500':   random.sample(math500_problems, 5),
    'omnimath':  random.sample(omnimath_problems, 5),
}
print(f'\nMini-samples ready: {sum(len(v) for v in mini.values())} total')

In [ ]:
# Save full samples to cache
for name, problems in [('humaneval', humaneval_problems), ('math500', math500_problems),
                       ('omnimath', omnimath_problems), ('apps', apps_problems)]:
    path = DATA_CACHE / f'{name}_sample.json'
    with open(path, 'w') as f:
        json.dump(problems, f, indent=2)
    print(f'Saved {len(problems)} → {path}')

---
## Section 4 — Evaluation Harnesses

### 4a. HumanEval — execution-based pass@1

In [ ]:
def eval_humaneval(generated_code: str, problem: dict, timeout: float = 5.0) -> dict:
    """
    Execute generated_code + problem['test'] in a subprocess.
    Returns {'passed': bool, 'error': str|None}.
    """
    # Build a self-contained script: generated code + test function + check() call
    script = f"""
{generated_code}

{problem['test']}

check({problem['entry_point']})
print('PASS')
"""
    try:
        result = subprocess.run(
            [sys.executable, '-c', script],
            capture_output=True, text=True, timeout=timeout,
        )
        passed = result.returncode == 0 and 'PASS' in result.stdout
        error = result.stderr.strip()[:300] if result.returncode != 0 else None
        return {'passed': passed, 'error': error}
    except subprocess.TimeoutExpired:
        return {'passed': False, 'error': f'Timeout ({timeout}s)'}
    except Exception as e:
        return {'passed': False, 'error': str(e)}

# Quick sanity check with the canonical solution
sample_p = humaneval_problems[0]
result = eval_humaneval(sample_p['canonical'], sample_p)
print(f'Canonical solution test: passed={result["passed"]}  error={result["error"]}')

### 4b. MATH-500 & Omni-MATH — exact-match / rule-based evaluator

In [ ]:
def normalize_math_answer(s: str) -> str:
    """
    Lightweight normalizer for math answers.
    Strips LaTeX wrappers, whitespace, trailing punctuation.
    """
    if not s:
        return ''
    # unwrap \boxed{...}
    m = re.search(r'\\boxed\{(.+?)\}', s, re.DOTALL)
    if m:
        s = m.group(1)
    # strip $, \, spaces, trailing punctuation
    s = s.strip().strip('$').strip('\\').strip().rstrip('.')
    # normalize fractions: \frac{a}{b} → a/b
    s = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', s)
    # remove spaces inside expressions
    s = re.sub(r'\s+', '', s)
    return s.lower()


def eval_math(model_output: str, reference_answer: str) -> dict:
    """
    Returns {'passed': bool, 'model_answer': str, 'ref_answer': str}.
    Tries: (1) extract \\boxed{} from output, (2) last number, (3) full string match.
    """
    model_norm = normalize_math_answer(model_output)
    ref_norm   = normalize_math_answer(reference_answer)
    passed = model_norm == ref_norm

    # Fallback: try extracting last number-like token from model output
    if not passed:
        numbers = re.findall(r'-?\d+(?:[./]\d+)?', model_norm)
        if numbers and numbers[-1] == re.findall(r'-?\d+(?:[./]\d+)?', ref_norm or '0')[-1:]:
            passed = True

    return {'passed': passed, 'model_answer': model_norm, 'ref_answer': ref_norm}


# Test normalizer
test_cases = [
    ('\\boxed{42}', '42', True),
    ('The answer is $\\frac{3}{4}$', '3/4', True),
    ('\\boxed{\\frac{1}{2}}', '1/2', True),
    ('7.5', '7.5', True),
    ('Wrong answer', '5', False),
]
print('Normalizer tests:')
for model_out, ref, expected in test_cases:
    r = eval_math(model_out, ref)
    status = '✓' if r['passed'] == expected else '✗'
    print(f'  {status}  "{model_out[:30]}" vs "{ref}" → passed={r["passed"]} (expected {expected})')

---
## Section 5 — DyTopo Evaluation Loop (Mock Mode)

Run DyTopo on the mini-samples (5 problems × 4 datasets) using mock mode.
This validates the full pipeline before switching to real LLM.

In [ ]:
def extract_code_from_output(text: str) -> str:
    """Pull the first Python code block from agent output."""
    m = re.search(r'```python\s*\n(.+?)```', text, re.DOTALL)
    if m:
        return m.group(1)
    m = re.search(r'```\s*\n(.+?)```', text, re.DOTALL)
    if m:
        return m.group(1)
    return text  # fallback: return as-is


async def run_problem(problem: dict, task_type: str, mock: bool = True) -> dict:
    """Run DyTopo on a single problem, return result dict."""
    cfg = DyTopoConfig(
        task_type=task_type,
        n_agents=4,
        tau=0.3 if task_type == 'code' else 0.4,
        max_rounds=5,
        mock_mode=mock,
    )
    rounds = []
    final_answer = None
    async for rr in run_dytopo(problem['problem'], cfg):
        rounds.append(rr)
        if rr.final_answer:
            final_answer = rr.final_answer

    return {
        'id': problem['id'],
        'dataset': problem['dataset'],
        'n_rounds': len(rounds),
        'final_answer': final_answer or '',
        'rounds_data': [r.to_dict() for r in rounds],
    }


async def evaluate_mini(mock=True):
    results = []
    for dataset, problems in mini.items():
        if not problems:
            continue
        task_type = 'code' if dataset in ('humaneval', 'apps') else 'math'
        print(f'\n--- {dataset} ({task_type}) ---')
        for prob in problems:
            r = await run_problem(prob, task_type, mock=mock)
            results.append(r)
            print(f'  {prob["id"]:20s}  rounds={r["n_rounds"]}  '
                  f'answer={r["final_answer"][:60]}')
    return results

print('Running DyTopo on mini-samples (mock mode)...')
mock_results = asyncio.run(evaluate_mini(mock=True))
print(f'\nTotal: {len(mock_results)} problems evaluated')

---
## Section 6 — Real LLM Evaluation (Ollama)

Switches mock=False and uses Ollama locally. The paper uses `qwen3:8b` via vLLM;
locally we use `qwen2.5:7b` or `llama3.1:8b` — both produce valid DyTopo outputs.

**Setup:**
```bash
# Install Ollama (Mac): brew install ollama && ollama serve &
# Pull a model:
ollama pull qwen2.5:7b        # ~4.7 GB, recommended
# OR
ollama pull llama3.1:8b       # ~4.7 GB
# OR (fastest, lower quality)
ollama pull qwen2.5:3b        # ~2.3 GB
```

In [ ]:
import httpx

OLLAMA_BASE = 'http://localhost:11434/v1'
OLLAMA_MODEL = 'qwen2.5:7b'  # change to whichever model you have pulled

async def check_ollama():
    try:
        async with httpx.AsyncClient(timeout=5.0) as client:
            r = await client.get('http://localhost:11434/api/tags')
            r.raise_for_status()
            models = [m['name'] for m in r.json().get('models', [])]
            return True, models
    except Exception as e:
        return False, str(e)

ok, info = asyncio.run(check_ollama())
if ok:
    print(f'✓ Ollama is running')
    print(f'  Available models: {info}')
    # Auto-select best available model
    preferred = ['qwen2.5:7b', 'qwen3:8b', 'llama3.1:8b', 'llama3:8b', 'qwen2.5:3b']
    for m in preferred:
        if any(m in im for im in info):
            OLLAMA_MODEL = m
            print(f'  Selected model: {OLLAMA_MODEL}')
            break
    else:
        print(f'  No preferred model found. Set OLLAMA_MODEL manually.')
    OLLAMA_AVAILABLE = True
else:
    print(f'✗ Ollama not running: {info}')
    print('  Start with: ollama serve &')
    print('  Continuing in mock mode only.')
    OLLAMA_AVAILABLE = False

In [ ]:
# Run 2 problems per dataset with real LLM (if Ollama available)
# Warning: each problem takes ~30-90s with qwen2.5:7b on CPU

if OLLAMA_AVAILABLE:
    mini_llm = {
        k: v[:2] for k, v in mini.items() if v
    }

    async def run_problem_llm(problem, task_type):
        cfg = DyTopoConfig(
            task_type=task_type,
            n_agents=4,
            tau=0.3 if task_type == 'code' else 0.4,
            max_rounds=3,
            mock_mode=False,
            llm_base_url=OLLAMA_BASE,
            llm_model=OLLAMA_MODEL,
            temperature=0.3,
            max_tokens=800,
        )
        rounds = []
        async for rr in run_dytopo(problem['problem'], cfg):
            rounds.append(rr)
            print(f'    Round {rr.round}: {len(rr.agents)} agents, '
                  f'density={sum(sum(row) for row in rr.adjacency)/(4*3):.0%}')
        return rounds

    llm_results = {}
    for dataset, problems in mini_llm.items():
        task_type = 'code' if dataset in ('humaneval', 'apps') else 'math'
        print(f'\n--- {dataset} (REAL LLM: {OLLAMA_MODEL}) ---')
        for prob in problems:
            print(f'  Problem: {prob["id"]}')
            print(f'  Prompt: {prob["problem"][:100]}...')
            t0 = time.time()
            try:
                rounds = asyncio.run(run_problem_llm(prob, task_type))
                elapsed = time.time() - t0
                final = rounds[-1].final_answer if rounds and rounds[-1].final_answer else 'N/A'
                llm_results[prob['id']] = {'rounds': len(rounds), 'answer': final, 'time': elapsed}
                print(f'  → {len(rounds)} rounds, {elapsed:.1f}s, answer: {str(final)[:80]}')
            except Exception as e:
                print(f'  ✗ Error: {e}')
else:
    print('Skipping real LLM evaluation (Ollama not available).')
    print('To enable: ollama serve & then ollama pull qwen2.5:7b')

---
## Section 7 — Full Evaluation Harness (Scale to Paper)

This cell is the production harness — run it with `FULL_EVAL=True` for full paper-scale evaluation.
Expected runtime on a local machine with Ollama (qwen2.5:7b, CPU):

| Dataset | N | Time/problem | Total |
|---------|---|-------------|-------|
| HumanEval | 164 | ~60s | ~2.7h |
| APPS-Competition | 100 | ~90s | ~2.5h |
| MATH-500 | 500 | ~45s | ~6.2h |
| Omni-MATH | 70 | ~60s | ~1.2h |

**GPU (e.g. RTX 3090/4090):** ~8-15x faster → total ~1-2h. Highly recommended.

In [ ]:
FULL_EVAL = False  # Set True to run full evaluation

async def run_full_dataset(problems, task_type, results_path, mock=False,
                           llm_base=OLLAMA_BASE, llm_model=OLLAMA_MODEL,
                           tau=0.3, max_rounds=5, resume=True):
    """
    Full evaluation loop with:
    - Resume: skips already-completed problems (reads results_path if exists)
    - Per-problem timeout (120s)
    - Progress bar
    - Saves incremental JSON results
    """
    # Load existing results for resume
    done_ids = set()
    all_results = []
    if resume and os.path.exists(results_path):
        with open(results_path) as f:
            all_results = json.load(f)
        done_ids = {r['id'] for r in all_results}
        print(f'  Resuming: {len(done_ids)}/{len(problems)} already done')

    cfg = DyTopoConfig(
        task_type=task_type,
        n_agents=4,
        tau=tau,
        max_rounds=max_rounds,
        mock_mode=mock,
        llm_base_url=llm_base,
        llm_model=llm_model,
        temperature=0.3,
        max_tokens=1200,
    )

    remaining = [p for p in problems if p['id'] not in done_ids]

    for prob in tqdm(remaining, desc=f'{task_type} eval'):
        t0 = time.time()
        try:
            rounds = []
            async for rr in run_dytopo(prob['problem'], cfg):
                rounds.append(rr.to_dict())
            final_answer = next(
                (r['final_answer'] for r in reversed(rounds) if r.get('final_answer')), ''
            )
            result = {
                'id': prob['id'],
                'dataset': prob['dataset'],
                'n_rounds': len(rounds),
                'final_answer': final_answer,
                'time_s': round(time.time() - t0, 2),
                'rounds': rounds,
            }
        except Exception as e:
            result = {
                'id': prob['id'],
                'dataset': prob['dataset'],
                'error': str(e),
                'time_s': round(time.time() - t0, 2),
            }

        all_results.append(result)

        # Incremental save
        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2)

    return all_results


if FULL_EVAL and OLLAMA_AVAILABLE:
    print('Starting full evaluation...')
    for dataset, problems, task_type, tau in [
        ('humaneval', humaneval_problems, 'code', 0.3),
        ('math500',   math500_problems,   'math', 0.3),
        ('omnimath',  omnimath_problems,  'math', 0.4),
    ] + ([('apps', apps_problems, 'code', 0.3)] if APPS_OK else []):
        results_path = DATA_CACHE / f'{dataset}_results.json'
        print(f'\n=== {dataset} ({len(problems)} problems) ===')
        asyncio.run(run_full_dataset(
            problems, task_type, results_path,
            mock=False, tau=tau, max_rounds=5,
        ))
else:
    print('Full eval disabled (FULL_EVAL=False or Ollama not available).')
    print('Set FULL_EVAL=True and ensure Ollama is running to begin.')

---
## Section 8 — Score Results & Compare to Paper

Run this after full evaluation to compute accuracy and compare against Table 2.

In [ ]:
def score_results(results_path, dataset, problems_by_id):
    if not os.path.exists(results_path):
        print(f'  No results file: {results_path}')
        return None

    with open(results_path) as f:
        results = json.load(f)

    passed = 0
    errors = 0
    details = []

    for r in results:
        if 'error' in r:
            errors += 1
            details.append({'id': r['id'], 'passed': False, 'error': r['error']})
            continue

        prob = problems_by_id.get(r['id'])
        if not prob:
            continue

        if dataset == 'humaneval':
            code = extract_code_from_output(r.get('final_answer', ''))
            eval_r = eval_humaneval(code, prob)
            ok = eval_r['passed']
        else:  # math500, omnimath
            eval_r = eval_math(r.get('final_answer', ''), prob.get('answer', ''))
            ok = eval_r['passed']

        if ok:
            passed += 1
        details.append({'id': r['id'], 'passed': ok, **eval_r})

    total = len(results)
    acc = passed / total * 100 if total > 0 else 0
    return {'dataset': dataset, 'total': total, 'passed': passed,
            'errors': errors, 'accuracy': acc, 'details': details}


# Score all available results
prob_maps = {
    'humaneval': {p['id']: p for p in humaneval_problems},
    'math500':   {p['id']: p for p in math500_problems},
    'omnimath':  {p['id']: p for p in omnimath_problems},
    'apps':      {p['id']: p for p in apps_problems},
}

print('Results scoring:')
scores = {}
for ds in ['humaneval', 'math500', 'omnimath', 'apps']:
    path = DATA_CACHE / f'{ds}_results.json'
    s = score_results(path, ds, prob_maps[ds])
    if s:
        scores[ds] = s
        print(f'  {ds:18s}: {s["passed"]}/{s["total"]} ({s["accuracy"]:.1f}%)  errors={s["errors"]}')

if not scores:
    print('  No results yet — run full evaluation first (Section 7).')

In [ ]:
# Compare to paper Table 2 (Qwen3-8B backbone, the closest to local qwen2.5:7b)
paper_qwen3 = {
    'HumanEval':  {'single_agent': 18.29, 'random_topo': 80.49, 'agentscope': 80.00, 'dytopo': 89.63},
    'APPS-Comp':  {'single_agent': 9.62,  'random_topo': 24.24, 'agentscope': 23.12, 'dytopo': 25.14},
    'MATH-500':   {'single_agent': 48.57, 'random_topo': 68.57, 'agentscope': 72.86, 'dytopo': 75.71},
    'Omni-MATH':  {'single_agent': 21.43, 'random_topo': 25.71, 'agentscope': 35.71, 'dytopo': 51.43},
}

fig, ax = plt.subplots(figsize=(11, 5))

datasets_plot = list(paper_qwen3.keys())
x = np.arange(len(datasets_plot))
width = 0.18
colors = {'single_agent': '#64748b', 'random_topo': '#f87171', 'agentscope': '#fbbf24', 'dytopo': '#4ade80'}
labels_map = {'single_agent': 'Single Agent', 'random_topo': 'Random Topology',
              'agentscope': 'AgentScope', 'dytopo': 'DyTopo (paper)'}

for i, (key, label) in enumerate(labels_map.items()):
    vals = [paper_qwen3[d][key] for d in datasets_plot]
    ax.bar(x + (i - 1.5) * width, vals, width, label=label,
           color=colors[key], alpha=0.85)

# Add local DyTopo results if available
ds_map = {'HumanEval': 'humaneval', 'APPS-Comp': 'apps', 'MATH-500': 'math500', 'Omni-MATH': 'omnimath'}
local_vals = [scores.get(ds_map[d], {}).get('accuracy', None) for d in datasets_plot]
has_local = any(v is not None for v in local_vals)
if has_local:
    ax.scatter(x, [v if v is not None else 0 for v in local_vals],
               marker='*', s=200, color='#38bdf8', zorder=5, label='DyTopo (local, this run)')

ax.set_xticks(x)
ax.set_xticklabels(datasets_plot, color='white', fontsize=10)
ax.set_ylabel('Accuracy (%)', color='white')
ax.set_title('DyTopo vs Baselines — Qwen3-8B / Qwen2.5-7B\n(bars = paper Table 2, stars = local evaluation)',
             color='white', fontsize=10)
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=8)
ax.set_facecolor('#0f1828')
fig.patch.set_facecolor('#0a0e1a')
ax.tick_params(colors='white')
ax.spines[:].set_color('#1e293b')
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('dytopo_paper_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: dytopo_paper_comparison.png')

---
## Section 9 — Local Feasibility Summary

In [ ]:
print('=' * 70)
print('LOCAL REPRODUCTION FEASIBILITY REPORT')
print('=' * 70)

print('''
DATASETS — all freely downloadable, no gating:

  HumanEval       164 problems  ~130 KB   ✓ load_dataset works instantly
  APPS-Comp       100/1000      ~130 MB   ✓ works locally (viewer disabled on HF)
  MATH-500        500 problems  <1 MB     ✓ trivially small
  Omni-MATH       70/4428       ~10 MB    ✓ metric: rule-based (no GPT-4o needed)

MODELS — paper uses vLLM with:
  MiMo-V2-Flash   proprietary API (mimo.ai)   ✗ not locally available
  GPT-oss-120B    proprietary API              ✗ not locally available
  Llama3-8B-Instruct   Meta open weights      ✓ ollama pull llama3.1:8b (~4.7 GB)
  Qwen3-8B        Alibaba open weights        ✓ ollama pull qwen2.5:7b  (~4.7 GB)

SEMANTIC ENCODER:
  all-MiniLM-L6-v2  sentence-transformers    ✓ auto-downloaded on first encode()
  Size: ~90 MB

COMPUTE ESTIMATE (local CPU, Ollama, qwen2.5:7b):
  HumanEval  164 × 3 rounds × 4 agents × ~5s/agent  ≈  3-4h
  MATH-500   500 × 5 rounds × 4 agents × ~5s/agent  ≈  10-14h
  Omni-MATH   70 × 5 rounds × 4 agents × ~5s/agent  ≈  1.5-2h
  APPS-Comp  100 × 5 rounds × 4 agents × ~5s/agent  ≈  2-3h
  TOTAL (CPU)  ≈  17-24h

COMPUTE ESTIMATE (GPU, e.g. RTX 3090, Ollama, qwen2.5:7b):
  ~10-15x faster → TOTAL ≈  1.5-2.5h

VERDICT:
  ✓  Open-source models (Llama3-8B, Qwen2.5-7B) fully reproducible locally
  ✓  All 4 datasets free to download
  ✓  Semantic encoder free (sentence-transformers)
  ✗  Proprietary models (MiMo-V2-Flash, GPT-oss-120B) require API access
  ⚠  CPU-only: feasible but slow; GPU strongly recommended for MATH-500
  ⚠  Expected accuracy gap vs paper: ~3-8pp lower (qwen2.5:7b vs qwen3:8b +
     inference infra differences)

RECOMMENDED LOCAL SETUP:
  ollama pull qwen2.5:7b
  ollama serve &
  # Then set FULL_EVAL=True in Section 7 above
  # Start with HumanEval (164 problems, fastest to iterate)
''')
print('=' * 70)